In [1]:
# Cell 1: Setup & Import
from google.cloud import aiplatform
from google.cloud import storage
from datetime import datetime
import pickle
import os

PROJECT_ID = "telco-portfolio"
LOCATION = "us-central1"
BUCKET_NAME = "telco-portfolio-bucket-20260602"  # Gunakan bucket terbaru

# Initialize Vertex AI
aiplatform.init(project=PROJECT_ID, location=LOCATION)

print(f"✅ Project: {PROJECT_ID}")
print(f"✅ Location: {LOCATION}")
print(f"✅ Bucket: {BUCKET_NAME}")

✅ Project: telco-portfolio
✅ Location: us-central1
✅ Bucket: telco-portfolio-bucket-20260602


In [2]:
# Cell 2: Find latest model in Cloud Storage
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

# List all model files
blobs = list(bucket.list_blobs(prefix="models/"))
model_files = [blob for blob in blobs if blob.name.endswith('.pkl') and 'latest' not in blob.name]

if model_files:
    # Get the latest model by creation time
    latest_model = sorted(model_files, key=lambda x: x.time_created, reverse=True)[0]
    model_gcs_path = f"gs://{BUCKET_NAME}/{latest_model.name}"
    model_uri = f"gs://{BUCKET_NAME}/models/"
    print(f"✅ Latest model found: {latest_model.name}")
    print(f"📁 Model URI: {model_uri}")
    print(f"📅 Created: {latest_model.time_created}")
else:
    print("❌ No model found in Cloud Storage")
    model_uri = f"gs://{BUCKET_NAME}/models/"
    print(f"📁 Using default model URI: {model_uri}")

✅ Latest model found: models/model_telco_churn_20260602_105004.pkl
📁 Model URI: gs://telco-portfolio-bucket-20260602/models/
📅 Created: 2026-06-02 03:50:12.950000+00:00


In [4]:
from google.cloud import storage

BUCKET_NAME = "telco-portfolio-bucket-20260602"
storage_client = storage.Client(project="telco-portfolio")
bucket = storage_client.bucket(BUCKET_NAME)

# Copy file ke model.pkl
source_blob = bucket.blob("models/latest_model.pkl")
destination_blob = bucket.blob("models/model.pkl")
bucket.copy_blob(source_blob, bucket, "models/model.pkl")

print("✅ model.pkl created in bucket")

✅ model.pkl created in bucket


In [5]:
# Cell 3: Upload Model to Vertex AI Model Registry
from google.cloud import aiplatform

PROJECT_ID = "telco-portfolio"
LOCATION = "us-central1"
BUCKET_NAME = "telco-portfolio-bucket-20260602"
model_uri = f"gs://{BUCKET_NAME}/models/"

aiplatform.init(project=PROJECT_ID, location=LOCATION)

model = aiplatform.Model.upload(
    display_name="telco-churn-predictor-v2",
    artifact_uri=model_uri,
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/xgboost-cpu.1-6:latest",
    serving_container_predict_route="/predict",
    serving_container_health_route="/health",
)

print(f"✅ Model uploaded successfully!")
print(f"🔗 Model resource name: {model.resource_name}")
print(f"📋 Model name: {model.name}")

Creating Model
Create Model backing LRO: projects/1087639212876/locations/us-central1/models/3867959310969470976/operations/6285132090153369600
Model created. Resource name: projects/1087639212876/locations/us-central1/models/3867959310969470976@1
To use this Model in another session:
model = aiplatform.Model('projects/1087639212876/locations/us-central1/models/3867959310969470976@1')
✅ Model uploaded successfully!
🔗 Model resource name: projects/1087639212876/locations/us-central1/models/3867959310969470976
📋 Model name: 3867959310969470976


In [6]:
# Cell 4: Create Endpoint
endpoint = aiplatform.Endpoint.create(
    display_name="telco-churn-endpoint-v2",
    project=PROJECT_ID,
    location=LOCATION,
)

print(f"✅ Endpoint created!")
print(f"🔗 Endpoint resource name: {endpoint.resource_name}")
print(f"📋 Endpoint name: {endpoint.name}")

Creating Endpoint
Create Endpoint backing LRO: projects/1087639212876/locations/us-central1/endpoints/1504880124460269568/operations/5812817079232888832
Endpoint created. Resource name: projects/1087639212876/locations/us-central1/endpoints/1504880124460269568
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/1087639212876/locations/us-central1/endpoints/1504880124460269568')
✅ Endpoint created!
🔗 Endpoint resource name: projects/1087639212876/locations/us-central1/endpoints/1504880124460269568
📋 Endpoint name: 1504880124460269568


In [7]:
# Cell 5: Deploy Model to Endpoint
model.deploy(
    endpoint=endpoint,
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1,
    traffic_percentage=100,
)

print(f"✅ Model deployed successfully!")
print(f"🔗 Endpoint ready at: {endpoint.name}")

Deploying model to Endpoint : projects/1087639212876/locations/us-central1/endpoints/1504880124460269568
Deploy Endpoint model backing LRO: projects/1087639212876/locations/us-central1/endpoints/1504880124460269568/operations/3049295757887668224


FailedPrecondition: 400 Model server never became ready. Please validate that your model file or container configuration are valid. Model server logs can be found at https://console.cloud.google.com/logs/viewer?project=1087639212876&resource=aiplatform.googleapis.com%2FEndpoint&advancedFilter=resource.type%3D%22aiplatform.googleapis.com%2FEndpoint%22%0Aresource.labels.endpoint_id%3D%221504880124460269568%22%0Aresource.labels.location%3D%22us-central1%22. 9: Model server never became ready. Please validate that your model file or container configuration are valid. Model server logs can be found at https://console.cloud.google.com/logs/viewer?project=1087639212876&resource=aiplatform.googleapis.com%2FEndpoint&advancedFilter=resource.type%3D%22aiplatform.googleapis.com%2FEndpoint%22%0Aresource.labels.endpoint_id%3D%221504880124460269568%22%0Aresource.labels.location%3D%22us-central1%22.